# Tutorial 06: Structured Multi-Agent Workflow Loop (Offline-First)

Deep tutorial for `A2A/06_Multi-Agent_Workflow`, focusing on state tracking, step planning, routing, payload building, and sequential invocation.

## 1) Where We Are in the Journey

`05_Negotiation` introduced execute-many and evaluate-best.
This step focuses on controlled sequential workflow with stateful chaining and orchestration components.
It exists to make the execution loop extensible and maintainable.

## 2) What You Will Learn

- Planner/router/executor separation
- State-carrying execution loop
- Payload construction that depends on prior state
- Differences between delegation and full workflow orchestration

## 3) Why This Matters

As tasks become multi-step, control logic must be explicit.
Workflow loops make failure handling, retries, and observability possible.
This folder bridges conceptual architecture notes and runnable orchestration mechanics.

## 4) Core Concept Deep Dive

Key idea: workflow = planning + state + controlled iteration.
From this folder: `plan(query)` builds typed steps, `build_payload(step, state)` injects dependencies, `execute(query)` runs route->invoke->state-update loop.
Trade-off: deterministic control is debuggable but requires careful schema discipline.

## 5) Code Walkthrough (Only `A2A/06_Multi-Agent_Workflow`)

- Early markdown/raw cells define architecture and component roles.
- `discover_capabilities()` creates registry.
- `plan()` creates finance/math step objects.
- `build_payload()` injects previous result for dependent `add` step.
- `execute()` routes by tool presence and updates state.

In [ ]:
import re

MOCK_REGISTRY = [
    {'agent': 'math-agent', 'endpoint': 'http://localhost:8000/invoke', 'tools': [{'name': 'add', 'description': 'Add two numbers'}]},
    {'agent': 'finance-agent', 'endpoint': 'http://localhost:8001/invoke', 'tools': [{'name': 'calculate_interest', 'description': 'Calculate simple interest'}]}
]

def discover_capabilities():
    return MOCK_REGISTRY

In [ ]:
def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
    steps = []
    if 'interest' in query:
        steps.append({'type': 'finance', 'tool': 'calculate_interest', 'args': {'principal': numbers[0], 'rate': numbers[1], 'time': numbers[2]}})
    if 'add' in query:
        steps.append({'type': 'math', 'tool': 'add', 'args': {'b': numbers[-1]}})
    return steps

def build_payload(step, state):
    args = dict(step['args'])
    if step['tool'] == 'add':
        args['a'] = state['result']
    return {'tool': step['tool'], 'args': args}

In [ ]:
def invoke_sim(agent, payload):
    if payload['tool'] == 'calculate_interest':
        a = payload['args']
        return {'status': 'success', 'result': (a['principal'] * a['rate'] * a['time']) / 100}
    if payload['tool'] == 'add':
        a = payload['args']
        return {'status': 'success', 'result': a['a'] + a['b']}
    return {'status': 'error', 'message': 'Unsupported tool'}

def execute(query):
    registry = discover_capabilities()
    steps = plan(query)
    state = {'result': None}
    for step in steps:
        agent = next(a for a in registry if step['tool'] in [t['name'] for t in a['tools']])
        payload = build_payload(step, state)
        response = invoke_sim(agent, payload)
        if response.get('status') == 'success':
            state['result'] = response['result']
        else:
            print('Step failed:', response)
            break
    return state['result']

In [ ]:
query = 'calculate interest for 2000 at 10 for 1 year and then add 500'
print('FINAL:', execute(query))

## 6) System Flow Explanation

1. Build capability registry.
2. Plan query into typed ordered steps.
3. Route each step by tool capability.
4. Build payload with state-aware arguments.
5. Invoke, validate, and update state.
6. Return final chained output.

## 7) Important Concepts You Might Miss

- Route decision is repeated per step, not once per query.
- State object is orchestration memory.
- Build-payload stage is where data dependency is encoded.

## 8) Common Mistakes / Pitfalls

- Missing result injection before dependent steps.
- Assuming every response has `status=success`.
- Ignoring tool-name/schema alignment across agents.

## 9) Key Takeaways

- This folder formalizes the workflow loop.
- State-aware payload construction enables dependency chaining.
- Structured orchestration is foundation for async and long-running patterns.

## 10) What’s Next (Strict Preview)

`07_Asynchronous _Long-Running_A2A` addresses tasks that cannot complete instantly.
It solves submission, deferred completion, and status tracking concerns.
This matters when execution latency is external or unpredictable.